# <center> Token Classification </center>

In this notebook, we will finetuning NLP model (in this case IndoBERT from IndoNLU) for token classification task, using transformers library by 🤗. More specifically, we will make the model capable to perform NER task. We gonna use dataset from [IndoNLU](https://github.com/indobenchmark/indonlu) called NERGrit. The labels of token in the NERGrit dataset consist of PERSON (name
of person), PLACE (name of location), and ORGA-
NIZATION (name of organization), which follow the IOB2 chunking format.

## Install the sentencepiece, transformers, and datasets library.

These are the necessary library we need to install, since these are not available by default in Google Colab.

In [ ]:
# !pip install sentencepiece==0.2.1
# !pip install transformers==5.11.0
# !pip install datasets==5.0.0
# !pip install seqeval
# !pip install tqdm==4.68.2
# !pip install accelerate==1.14.0


## Import depedency library

In [1]:
from pathlib import Path
import numpy as np
import re
import os
from datasets import Dataset
import evaluate 
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForTokenClassification, BertTokenizerFast, TrainingArguments, Trainer, DataCollatorForTokenClassification, AutoConfig
from sklearn.model_selection import train_test_split
import nltk
from tqdm.auto import tqdm
from seqeval.scheme import IOB2
from seqeval.metrics import classification_report as seqeval_classification_report
nltk.download('punkt_tab')
# if error related to tqdm happens, just run the previous cell once again

# set enviroment for tracking report folder
os.environ["TENSORBOARD_LOGGING_DIR"] ="./tensorboard_report"

[nltk_data] Downloading package punkt_tab to /home/hadji-
[nltk_data]     musfushi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Data

### Download data

In [2]:
folder_data_path = "data" 


In [ ]:
!mkdir {folder_data_path}

# download train dataset
!wget -P {folder_data_path} "https://raw.githubusercontent.com/indobenchmark/indonlu/master/dataset/nergrit_ner-grit/train_preprocess.txt"

# download test dataset, note that we actually use the validation dataset from the repository, since the real test dataset is masked
!wget -P {folder_data_path} "https://raw.githubusercontent.com/indobenchmark/indonlu/master/dataset/nergrit_ner-grit/valid_preprocess.txt"

# you can see the downloaded files by clicking the refresh icon on te left, the files is in the form of txt file.

### Read data

In [3]:
# function to read data
def read_wnut(file_path):
    file_path = Path(file_path)

    raw_text = file_path.read_text().strip()
    raw_docs = re.split(r'\n\t?\n', raw_text)
    token_docs = []
    tag_docs = []
    for doc in raw_docs:
        tokens = []
        tags = []
        for line in doc.split('\n'):
            token, tag = line.split('\t')
            tokens.append(token)
            tags.append(tag)
        token_docs.append(tokens)
        tag_docs.append(tags)

    return token_docs, tag_docs

In [4]:
# read data
train_texts, train_tags = read_wnut(f'{folder_data_path}/train_preprocess.txt')
test_texts, test_tags = read_wnut(f'{folder_data_path}/valid_preprocess.txt')

Here some snippet of the content of the txt file



In [5]:
train_index = 12
print(train_texts[train_index], train_tags[train_index], sep='\n')

['Telah', 'menjadi', 'universitas', 'sejak', '1966', ',', 'tetapi', 'merupakan', 'institusi', 'sejak', '1909', ',', 'ketika', 'Loughborough', 'Technical', 'Institute', 'didirikan', 'dengan', 'fokus', 'terhadap', 'kemampuan', 'dan', 'pengetahuan', 'yang', 'dapat', 'diaplikasikan', 'di', 'seluruh', 'dunia', ',', 'sebuah', 'tradisi', 'yang', 'masih', 'berlanjut', 'hingga', 'sekarang', ',', 'dengan', 'UNIEI', 'mendanai', '"', 'Survei', 'Tahunan', 'terhadap', 'Aktivitas', 'Perpindahan', 'Teknologi', 'Universitas', '"', 'yang', 'menjadikan', 'Loughborough', 'sebagai', 'operasi', 'perpindahan', 'teknologi', 'paling', 'efisien', 'di', 'Britania', '.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORGANISATION', 'I-ORGANISATION', 'I-ORGANISATION', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORGANISATION', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PLACE', 'O', 'O', 'O', 'O', 'O',

In [6]:
test_index = 12
print(test_texts[test_index], test_tags[test_index], sep='\n')

['Selain', 'itu', 'juga', 'ada', 'penerbangan', 'reguler', 'di', 'Bandara', 'Bersujud', ',', 'pada', 'tahun', '2007', 'dilayani', 'oleh', 'PT', '.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PLACE', 'I-PLACE', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


As you can see, the sentence already split into words the labels is aligned with the word tokens. If you use custom data with different form, you need to process it into the form like above.

### Train val split

Next we will split loaded train data into train and validation data

In [7]:
train_texts, val_texts, train_tags, val_tags = train_test_split(train_texts, train_tags, test_size=0.2, random_state=42)

# this code below for test purpose, since we want to test the code with small number of data

# train_texts = train_texts[:100]
# val_texts = val_texts[:100]
# train_tags = train_tags[:100]
# val_tags = val_tags[:100]

In [8]:
print(f"Total train data: {len(train_texts)}, {len(train_tags)}")
print(f"Total validation data: {len(val_texts)}, {len(val_tags)}")
print(f"Total test data: {len(test_texts)}, {len(test_tags)}")

Total train data: 1337, 1337
Total validation data: 335, 335
Total test data: 209, 209


Create the mapper for label

In [9]:
unique_tags = set(tag for doc in train_tags + val_tags for tag in doc)
tag2id = {tag: id for id, tag in enumerate(unique_tags)}
id2tag = {id: tag for tag, id in tag2id.items()}

In [10]:
tag2id

{'B-ORGANISATION': 0,
 'I-ORGANISATION': 1,
 'B-PLACE': 2,
 'I-PERSON': 3,
 'O': 4,
 'I-PLACE': 5,
 'B-PERSON': 6}

In [11]:
id2tag 

{0: 'B-ORGANISATION',
 1: 'I-ORGANISATION',
 2: 'B-PLACE',
 3: 'I-PERSON',
 4: 'O',
 5: 'I-PLACE',
 6: 'B-PERSON'}

In [23]:
# create the folder to put idtag files
!mkdir idtag

mkdir: idtag: File exists


Save the mapper into folder idtag

In [12]:
idtag_dir = "idtag"

# !!! don't forget to save the label map, so that we can use it to translate the model output, we will put these into the "idtag" folder
with open(f"{idtag_dir}/tag2id_pkl", 'wb') as f1:
    pickle.dump(tag2id, f1)
    
with open(f'{idtag_dir}/id2tag_pkl', 'wb') as f2:
    pickle.dump(id2tag, f2)  

### Preprocess data

#### Encode input text data

Load Tokenizer to preprocess data

In [13]:
tokenizer_checkpoint = "indobenchmark/indobert-lite-base-p1" 
tokenizer = BertTokenizerFast.from_pretrained(tokenizer_checkpoint, do_lower_case=True)

Encode text data

In [14]:
encoder_max_len = 300 # 512 is the maximum number of token input, you can modify this value, but i will take 300 for now
train_encodings = tokenizer(train_texts, is_split_into_words=True, return_offsets_mapping=True, max_length=encoder_max_len, truncation=True, padding=False)
val_encodings = tokenizer(val_texts, is_split_into_words=True, return_offsets_mapping=True, max_length=encoder_max_len, truncation=True, padding=False)

#### Encode label

In [15]:
# function to encode label data
def encode_tags(tags, encodings):
    labels = [[tag2id[tag] for tag in doc] for doc in tags]
    encoded_labels = []
    for doc_labels, doc_offset in zip(labels, encodings.offset_mapping):
        # create an empty array of -100
        doc_enc_labels = np.ones(len(doc_offset),dtype=int) * -100
        arr_offset = np.array(doc_offset)

        # set labels whose first offset position is 0 and the second is not 0
        doc_enc_labels[(arr_offset[:,0] == 0) & (arr_offset[:,1] != 0)] = doc_labels
        encoded_labels.append(doc_enc_labels.tolist())

    return encoded_labels

In [16]:
# encode labels
train_labels = encode_tags(train_tags, train_encodings)
val_labels = encode_tags(val_tags, val_encodings)

Before continuing we will calculate the class weight here, in case we want to factor the class imbalance in custom loss function later.

In [17]:
def calculate_class_weights(train_labels):

    unrolled_train_labels = [element for label in train_labels for element in label if element != -100]
    unrolled_train_labels = torch.tensor(unrolled_train_labels)
    class_count = torch.bincount(unrolled_train_labels)
    class_weights = 1.0 / class_count
    class_weights = class_weights / class_weights.sum()
    return class_weights 

In [18]:
class_weights = calculate_class_weights(train_labels)
class_weights 

tensor([0.2421, 0.2052, 0.0931, 0.1910, 0.0053, 0.1086, 0.1547])

#### Wrap data using custom train and validation set with dataset class

In [19]:
class WNUTDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [20]:
train_encodings.pop("offset_mapping") # we don't want to pass this to the model
val_encodings.pop("offset_mapping")
train_dataset = WNUTDataset(train_encodings, train_labels)
val_dataset = WNUTDataset(val_encodings, val_labels)

## Training

we will use Trainer class from transformers library for training

In [21]:
# Create data collator, this will automatically handle how we will treat the batch, 
# in this case we do the padding for each batch following the longest sequence in the batch

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [22]:
train_batch = 4 # batch size per device during training
eval_batch = 4 # batch size for evaluation
logging_steps = int(np.floor(len(train_texts)/train_batch)) # train loss is logged every epoch

training_args = TrainingArguments(
    output_dir='./results',             # output directory
    eval_strategy = "epoch",
    num_train_epochs = 3,               # total number of training epochs
    learning_rate = 2e-5,
    per_device_train_batch_size = train_batch,  
    per_device_eval_batch_size = eval_batch,   
    weight_decay = 0.01,                # strength of weight decay
    report_to = 'tensorboard',             # directory for storing logs
    logging_steps= logging_steps,
    load_best_model_at_end = True,      # load the best model after the end of training
    metric_for_best_model = 'eval_f1',
    greater_is_better = True,
    save_total_limit = 1,               # save only one model
    dataloader_drop_last = True,
    save_strategy = 'epoch'             # the value must be same with eval_strategy
)

# check for the docs for the explanation of the other parameter, be diligent lads!

In [23]:
model_checkpoint =tokenizer_checkpoint

config = AutoConfig.from_pretrained(
    model_checkpoint,
    _num_labels=len(unique_tags),
    id2label=id2tag,
    label2id=tag2id
)

model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, config=config)

Loading weights:   0%|          | 0/23 [00:00<?, ?it/s]

[transformers] AlbertForTokenClassification LOAD REPORT from: indobenchmark/indobert-lite-base-p1
Key               | Status     | 
------------------+------------+-
pooler.weight     | UNEXPECTED | 
pooler.bias       | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
# need to create custom metrics for token classification task, it used seqeval metric normally used in 

# using seqeval metric from datasets library
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens) where the code is -100
    true_predictions = [
        [id2tag[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2tag[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

#### Create Custom Loss

We can use custom loss for training, in this example we will use Focal Loss 

In [25]:
class TokenFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, ignore_index=-100, reduction='mean'):
        """
        Multi-class Focal Loss for Token Classification.
        
        Args:
            alpha (Tensor, optional): A manual rescaling weight given to each class.
                                      Shape should be (num_classes,).
            gamma (float): Focusing parameter. Higher values down-weight easy tokens more.
            ignore_index (int): Specifies a target value that is ignored (e.g., -100 for pad tokens).
            reduction (str): 'mean', 'sum', or 'none'.
        """
        super(TokenFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits shape: (batch_size, sequence_length, num_classes)
        # targets shape: (batch_size, sequence_length)
        
        # 1. Flatten tensors for element-wise calculation
        num_classes = logits.size(-1)
        logits = logits.view(-1, num_classes)
        targets = targets.view(-1)

        # 2. Create mask to filter out ignored tokens (like padding or special tokens)
        valid_mask = (targets != self.ignore_index)
        
        # If there are no valid tokens in the batch, return zero loss
        if not valid_mask.any():
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        # Filter active logits and targets
        active_logits = logits[valid_mask]
        active_targets = targets[valid_mask]

        # 3. Calculate Cross Entropy base probabilities (pt)
        # log_softmax is more numerically stable than softmax
        log_p = F.log_softmax(active_logits, dim=-1)
        
        # Gather the log probabilities of the true target classes
        log_pt = log_p.gather(dim=-1, index=active_targets.unsqueeze(1)).squeeze(1)
        p_t = torch.exp(log_pt)

        # 4. Calculate the Focal Loss modulation factor
        focal_weight = (1 - p_t) ** self.gamma
        loss = -focal_weight * log_pt

        # 5. Apply Alpha class weights if provided
        if self.alpha is not None:
            # Ensure alpha is on the correct device
            self.alpha = self.alpha.to(logits.device)
            # Gather alpha weight for each target token
            alpha_t = self.alpha.gather(dim=0, index=active_targets)
            loss = alpha_t * loss

        # 6. Apply reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

Create new Trainer class to inject the custom loss object

In [26]:
class FocalLossTrainer(Trainer):

    def __init__(self, loss_fct, **kwargs):
        super().__init__(**kwargs)
        self.loss_fct = loss_fct

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        labels = inputs.get("labels")
        
        # Calculate custom loss
        loss = self.loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

In [27]:
focal_loss = True

if focal_loss:
    # instantiate focal loss funtion
    loss_fct = TokenFocalLoss(alpha=class_weights)
    trainer = FocalLossTrainer(
        loss_fct=loss_fct,                   # inject custom loss function
        model=model,                         # the instantiated 🤗 Transformers model to be trained
        args=training_args,                  # training arguments, defined above
        train_dataset=train_dataset,         # training dataset
        eval_dataset=val_dataset,            # evaluation dataset
        compute_metrics=compute_metrics,     # custom metrics  
        processing_class = tokenizer,
        data_collator =  data_collator,
    )
else:
    trainer = Trainer(
        model=model,                         # the instantiated 🤗 Transformers model to be trained
        args=training_args,                  # training arguments, defined above
        train_dataset=train_dataset,         # training dataset
        eval_dataset=val_dataset,            # evaluation dataset
        compute_metrics=compute_metrics,     # custom metrics  
        processing_class = tokenizer,
        data_collator =  data_collator
    )

In [28]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.013198,0.007335,0.434783,0.696143,0.535262,0.860359
2,0.004838,0.006363,0.546693,0.793039,0.647217,0.901479
3,0.003056,0.006371,0.576871,0.797742,0.669562,0.908736


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1002, training_loss=0.0070304638611342375, metrics={'train_runtime': 150.5037, 'train_samples_per_second': 26.651, 'train_steps_per_second': 6.658, 'total_flos': 11154833123040.0, 'train_loss': 0.0070304638611342375, 'epoch': 3.0})

In [30]:
# we can see that only the best model is loaded in trainer object at the end
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.003056,0.006371,3,0.576871,0.797742,0.669562,0.908736


{'eval_loss': 0.006370695307850838,
 'eval_precision': 0.5768707482993197,
 'eval_recall': 0.7977422389463782,
 'eval_f1': 0.6695617844453217,
 'eval_accuracy': 0.908735696343846}

## Using the model

In [31]:
# load the best model, make sure the best model file location is the right one, check the folder results 
best_model_checkpoint = "results/checkpoint-1002"
best_model = AutoModelForTokenClassification.from_pretrained(best_model_checkpoint)

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

In [32]:
# set get device function
def get_default_device():
    """Pick GPU if available, else CPU"""
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

In [33]:
# get device
device = get_default_device()

In [34]:
# put the model into device (GPU in case we use GPU)
best_model = best_model.to(device)

In [35]:
# define predict function, this function takes a sentence and return word tokens of the sentence with its label
def best_model_predict(text):

  word_token = nltk.word_tokenize(text)
  tokenized_input_text = tokenizer(word_token, is_split_into_words=True, return_offsets_mapping=True, max_length=encoder_max_len, padding='max_length', truncation=True, 
                                   return_tensors="pt")
  input_ids, attention_mask, offset_mapping = tokenized_input_text["input_ids"], tokenized_input_text["attention_mask"], tokenized_input_text["offset_mapping"]
  input_ids = input_ids.to(device)
  attention_mask = attention_mask.to(device)
  output = best_model(input_ids = input_ids, attention_mask = attention_mask)
  logits = output.logits
  logits_labels = torch.argmax(logits, dim=-1).tolist()
  tag = np.array([model.config.id2label[i] for i in logits_labels[0]])
  # since by the tokenizer some word are splitted, so we need to map the output label using the offset_mapping into the right token
  arr_offset = offset_mapping.numpy()[0]
  mask = (arr_offset[:,0] == 0) & (arr_offset[:,1] != 0)

  return word_token, tag[mask].tolist()

In [36]:
# Test the function
text = "Ketua KPK, Firli Bahuri, meminta dugaan pemberian uang Rp 650 juta ke orang mengaku sebagai pegawai KPK dari mantan Bupati Kuantan Singingi (Kuansing) Mursini dibuktikan. "

tokenized_text, predicted_label = best_model_predict(text)

In [37]:
# print the resul
for word, label in zip(tokenized_text, predicted_label):
  print(word, " --> ", label)

Ketua  -->  O
KPK  -->  B-ORGANISATION
,  -->  O
Firli  -->  B-PERSON
Bahuri  -->  I-PERSON
,  -->  O
meminta  -->  O
dugaan  -->  O
pemberian  -->  O
uang  -->  O
Rp  -->  O
650  -->  O
juta  -->  O
ke  -->  O
orang  -->  O
mengaku  -->  O
sebagai  -->  O
pegawai  -->  O
KPK  -->  B-ORGANISATION
dari  -->  O
mantan  -->  O
Bupati  -->  O
Kuantan  -->  B-PLACE
Singingi  -->  I-PLACE
(  -->  O
Kuansing  -->  B-PERSON
)  -->  O
Mursini  -->  B-PERSON
dibuktikan  -->  O
.  -->  O


The model performance is quite good.

## Prediction and evaluation on test data!

Now, lets do some prediction on test data and evaluate the result, but without using trainer class. We will using Dataset class from transformers lib and DataLoader from pytorch lib since we want to utilize GPU optimaly. 

In [38]:
batch_size = 4
test_encoding = tokenizer(test_texts, is_split_into_words=True, return_offsets_mapping=True, max_length=encoder_max_len, padding='max_length', truncation=True)
offset_mapping = test_encoding.pop('offset_mapping')
test_dataset = Dataset.from_dict({'input_ids': test_encoding['input_ids'], 'attention_mask': test_encoding['attention_mask']})
# we need to format the data into pytorch Tensor, since we use pytorch model for prediction
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'], output_all_columns=True)
# put the dataset into data loader
test_dl = DataLoader(test_dataset, batch_size, shuffle=False)

In [39]:
# define predict function, this function takes a sentence and return word tokens of the sentence with its label
def best_model_predict_test_data(data_loader, offset_mapping):

  pbar = tqdm(data_loader, leave=True, total=len(data_loader))

  with torch.no_grad():
    list_pred_label = []
    for idx, data in enumerate(pbar):
      input_ids, attention_mask = data["input_ids"], data["attention_mask"]
      input_ids = input_ids.to(device)
      attention_mask = attention_mask.to(device)
      output = best_model(input_ids = input_ids, attention_mask = attention_mask)
      logits = output.logits
      logits_labels = torch.argmax(logits, dim=-1).tolist()

      for label in logits_labels:
        tag = np.array([id2tag[i] for i in label])
        list_pred_label.append(tag)

  final_list_pred_label = []
  for pred_label, map in zip(list_pred_label, offset_mapping):
      arr_map = np.array(map)
      mask = (arr_map[:,0] == 0) & (arr_map[:,1] != 0)
      final_list_pred_label.append(pred_label[mask].tolist())

  return final_list_pred_label

In [40]:
pred_test_labels = best_model_predict_test_data(test_dl, offset_mapping)

  0%|          | 0/53 [00:00<?, ?it/s]

Lets do some evaluation

In [41]:
print(seqeval_classification_report(test_tags, pred_test_labels, mode='strict', scheme=IOB2))

              precision    recall  f1-score   support

ORGANISATION       0.40      0.81      0.53       121
      PERSON       0.70      0.83      0.76       213
       PLACE       0.72      0.84      0.77       328

   micro avg       0.62      0.83      0.71       662
   macro avg       0.60      0.82      0.69       662
weighted avg       0.65      0.83      0.72       662



The evaluation result is very nice

## Prediction using HF pipeline

In [42]:
from transformers import pipeline

model_checkpoint = "results/checkpoint-1002"
pipe = pipeline("token-classification", model=model_checkpoint, aggregation_strategy="max")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

In [43]:
input_text = "Ketua KPK, Firli Bahuri, meminta dugaan pemberian uang Rp 650 juta ke orang mengaku sebagai pegawai KPK dari mantan Bupati Kuantan Singingi (Kuansing) Mursini dibuktikan."
word_token = nltk.word_tokenize(input_text)
result = pipe(word_token, is_split_into_words=True)

In [44]:
encoding_out = tokenizer(word_token, is_split_into_words=True, return_offsets_mapping=True, max_length=encoder_max_len, truncation=True, padding=False)

In [45]:
tokenizer.decode(encoding_out)

'[CLS] ketua kpk, firli bahuri, meminta dugaan pemberian uang rp 650 juta ke orang mengaku sebagai pegawai kpk dari mantan bupati kuantan singingi ( kuansing ) mursini dibuktikan. [SEP]'

In [46]:
result

[[{'entity_group': 'ORGANISATION',
   'score': np.float32(0.7482694),
   'word': 'kpk',
   'start': 6,
   'end': 9},
  {'entity_group': 'PERSON',
   'score': np.float32(0.90085495),
   'word': 'firli bahuri',
   'start': 12,
   'end': 24},
  {'entity_group': 'ORGANISATION',
   'score': np.float32(0.68388796),
   'word': 'kpk',
   'start': 102,
   'end': 105},
  {'entity_group': 'PLACE',
   'score': np.float32(0.8814974),
   'word': 'kuantan singingi',
   'start': 125,
   'end': 141},
  {'entity_group': 'PERSON',
   'score': np.float32(0.739626),
   'word': 'kuansing',
   'start': 144,
   'end': 152},
  {'entity_group': 'PERSON',
   'score': np.float32(0.8040985),
   'word': 'mursini',
   'start': 155,
   'end': 162}]]

In [48]:
for e in result[0]:
    print(e["word"], " --> ", e["entity_group"])

kpk  -->  ORGANISATION
firli bahuri  -->  PERSON
kpk  -->  ORGANISATION
kuantan singingi  -->  PLACE
kuansing  -->  PERSON
mursini  -->  PERSON


## Download model

If you want to download the model run these code below

In [ ]:
!zip -r ./results.zip ./results

In [ ]:
!zip -r ./idtag.zip ./idtag

In [ ]:
from google.colab import files

In [ ]:
files.download("./results.zip")
files.download("./idtag.zip ")

***Author: Hadi Muhshi***